## IMPORTACION
Librerias, funciones, conexion

In [96]:
import mysql.connector
import pandas as pd
from datetime import datetime
import calendar
from dateutil.relativedelta import relativedelta
from pathlib import Path


In [ ]:
from utils import (
    obtener_inicio_mes_siguiente, #Inicio-fin del mes siguiente
    ejecutar_y_guardar,
    obtener_mes_anterior,  #Inicio-fin mes
    guardar_csv,
    obtener_quincena_anterior 
)

In [97]:
conn = mysql.connector.connect(
    host = "datamart.mex.internal.lyftbikes.com",
    port =3306,
    database= "mex_datawarehouse_bss4",
    user= "dm_mex_ge",
    password= "m+y#J9JI9F+^4qjSJLu^R",
)

conn.cursor().execute(
    "SET SESSION max_execution_time = 300000"
)

## FECHA INICIO Y FIN
Fechas que esta tomando en cuenta la consulta

In [ ]:
fecha_inicioQ, fecha_finQ = obtener_quincena_anterior()

print(fecha_inicioQ, fecha_finQ)

2026-06-01 00:00:00
2026-06-16 00:00:00


### Ruta
Agregar ruta y nombre del archivo

In [ ]:
ruta_archivo_quincenal = (
    rf"C:\Users\tsunami.martinez\Downloads\Reportes"
    rf"\Reporte_quincenal_{fecha_inicioQ[:7]}.csv"
)

print(ruta_archivo_quincenal)

## Query reporte quincenal

In [100]:
query_quincenal = f"""

SELECT  CONCAT(m.firstName, " ", m.lastName) AS nombre_completo,
        m.bikeMemberAttributes_gender AS genero,
        DATE(CONVERT_TZ(FROM_UNIXTIME(m.bikeMemberAttributes_birthday/1000), 'UTC', 'America/Costa_Rica')) AS fecha_nacimiento,
        m.address AS domicilio,
        m.bikeMemberAttributes_postalCode AS cp,
        m.email AS correo_electronico, 
        m.phoneNumber AS tel_contacto,
        IF(ss.id = 1, 'sitio_web', IF(ss.id = 3, 'renov_auto', IF(ss.id = 6, 'app_movil', 'otra'))) AS forma_inscripcion,
        m.bikeMemberAttributes_memberNumber AS num_pers_usuaria,
        m.currentTransitCardNumber AS num_tarjeta,
        st.name_localizedValue1 AS tipo_membresia

FROM BikeSubscriptionFact AS s
LEFT JOIN BikeMemberFact AS m ON s.member_accountNumber = m.bikeMemberAttributes_accountNumber
LEFT JOIN BikeSubscriptionTypeDim AS st ON s.subscriptionType_id = st.id
LEFT JOIN BikeSubscriptionRequestSourceDim AS ss ON s.subscriptionSourceDim_id = ss.id

WHERE s.start BETWEEN UNIX_TIMESTAMP(CONVERT_TZ('{fecha_inicioQ}', 'America/Costa_Rica', 'UTC'))*1000
  AND UNIX_TIMESTAMP(CONVERT_TZ('{fecha_finQ}', 'America/Costa_Rica', 'UTC'))*1000
  AND (s.subscriptionType_id < 6 OR s.subscriptionType_id = 9 ) -- sólo anuales, migradas, temporales y plus
;

"""

In [103]:
df_quincenal = ejecutar_y_guardar(
    query_quincenal,
    conn,
    ruta_archivo_quincenal
)

c:\Users\tsunami.martinez\Documents\Querys_Btk\utils.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


Archivo guadado: C:\Users\tsunami.martinez\Downloads\Reportes\Reporte_quincenal_2026-06.csv
